In [0]:
from pyspark.sql import SparkSession, functions as f
import time

spark = SparkSession.builder.getOrCreate()

spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.shuffle.partitions", "200")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

N_WINDOWS = 1_000
N_EVENTS = 25_000_000
TS_RANGE = N_WINDOWS * 100

results = {}

def fingerprint(df):
    """Order-independent summary of an (event_id, window_id) result, plus wall-clock seconds."""
    start = time.perf_counter()
    row = df.agg(
        f.count("*").alias("rows"),
        f.sum(f.pmod(f.xxhash64("event_id", "window_id"), f.lit(1_000_000_007))).alias("checksum"),
    ).first()
    seconds = time.perf_counter() - start
    return {"rows": row["rows"], "checksum": row["checksum"], "seconds": round(seconds, 2)}

In [0]:
windows = (
    spark.range(N_WINDOWS)
    .withColumnRenamed("id", "window_id")
    .withColumn("start_ts", f.col("window_id") * (TS_RANGE // N_WINDOWS))
    .withColumn("end_ts", f.col("start_ts") + 10 + f.pmod(f.xxhash64("window_id"), f.lit(51)))
)

events = (
    spark.range(N_EVENTS)
    .withColumnRenamed("id", "event_id")
    .withColumn("ts", f.pmod(f.xxhash64("event_id"), f.lit(TS_RANGE)))
)

windows_exploded = windows.select(
    "window_id",
    f.explode(f.sequence("start_ts", "end_ts")).alias("ts_bucket"),
)
 
print("windows rows:         ", windows.count())
print("windows_exploded rows:", windows_exploded.count())

In [0]:
bnlj = (events
    .join(
        f.broadcast(windows),
        on=(f.col("ts") >= f.col("start_ts")) & (f.col("ts") <= f.col("end_ts")),
        how="inner")
    .select("event_id", "window_id")
)

bnlj.explain()

results["broadcast_nested_loop"] = fingerprint(bnlj)

In [0]:
bhj = (events
    .join(
        f.broadcast(windows_exploded),
        on=f.col("ts") == f.col("ts_bucket"),
        how="inner",)
    .select("event_id", "window_id")
)
 
bhj.explain()

results["broadcast_hash"] = fingerprint(bhj)

In [0]:
shj = (events
    .join(
        windows_exploded.hint("shuffle_hash"),
        on=f.col("ts") == f.col("ts_bucket"),
        how="inner",)
    .select("event_id", "window_id")
)
 
shj.explain()

results["shuffle_hash"] = fingerprint(shj)

In [0]:
smj = (events
    .join(
        windows_exploded.hint("merge"),
        on=f.col("ts") == f.col("ts_bucket"),
        how="inner")
    .select("event_id", "window_id")
)
 
smj.explain()

results["sort_merge"] = fingerprint(smj)

In [0]:
import json
print(json.dumps(results, indent=2))